# DDR Chart Generator — Training Notebook

**Before running: Runtime → Change runtime type → T4 GPU**

### Assumed setup
Your Google Drive already has StepMania packs uploaded at:
```
MyDrive/ddc_project/data/raw/
  Some Pack Name/
    Song Name/
      Song.sm
      Song.mp3
  Another Pack/
    ...
```
Folder names don't matter. The code searches recursively for any folder
containing both a .sm file and an audio file (.mp3 / .ogg / .wav).

### Run order
**Every session:** re-run cells 1, 2, 3 (Colab VMs are wiped on disconnect).
**First time only:** run cells 4 onwards. Results save to Drive automatically.
**Resuming after disconnect:** re-run cells 1-3, then jump straight to cell 5 with `--curriculum_start` set to the last completed stage.

In [ ]:
# ── 1. Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ddc_project/data/cache
!mkdir -p /content/drive/MyDrive/ddc_project/data/raw
!mkdir -p /content/drive/MyDrive/ddc_project/checkpoints
!mkdir -p /content/drive/MyDrive/ddc_project/logs
print('Drive mounted.')

In [ ]:
# ── 2. Clone repo and symlink data ──────────────────────────────────────
# Replace with your actual GitHub repo URL
%cd /content
!git clone https://github.com/alexseungum/ieor142b_project.git ddc-chart-gen
%cd /content/ddc-chart-gen

# Symlink Drive data folder so code finds it at 'data/'
!ln -sf /content/drive/MyDrive/ddc_project/data data
print('Repo cloned and data symlinked.')

In [ ]:
# ── 3. Install dependencies ─────────────────────────────────────────────
!pip install librosa soundfile torchaudio --quiet
print('Done.')

In [ ]:
# ── 4. Verify data ───────────────────────────────────────────────────────
import sys, re
from pathlib import Path
from collections import defaultdict

sys.path.insert(0, '/content/ddc-chart-gen')

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
root = Path(DATA_ROOT)
audio_exts = {'.ogg', '.mp3', '.wav'}

def quick_difficulties(chart_path: str):
    """Fast grep for dance-single difficulty tags — no full parse needed."""
    try:
        content = open(chart_path, 'r', encoding='utf-8', errors='ignore').read()
        is_ssc = chart_path.endswith('.ssc')
        diffs = []
        if is_ssc:
            types = re.findall(r'#STEPSTYPE\s*:([^;]+);', content, re.IGNORECASE)
            diffd = re.findall(r'#DIFFICULTY\s*:([^;]+);', content, re.IGNORECASE)
            for t, d in zip(types, diffd):
                if 'single' in t.lower():
                    diffs.append(d.strip().lower())
        else:
            for m in re.finditer(r'#NOTES\s*:(.*?)(?=\n[^,\n]|\Z)', content,
                                  re.DOTALL | re.IGNORECASE):
                block = m.group(1)
                lines = [l.strip().rstrip(':') for l in block.split('\n')
                         if l.strip() and not l.strip().startswith('//')]
                if len(lines) >= 3 and 'single' in lines[0].lower():
                    diffs.append(lines[2].lower())
        return diffs
    except Exception:
        return []

all_sm  = list(root.rglob('*.sm'))
all_ssc = list(root.rglob('*.ssc'))
song_dirs = sorted({f.parent for f in all_sm + all_ssc})

print(f"{'='*60}")
print(f"DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total song directories found : {len(song_dirs)}")
print(f"  .sm  files                 : {len(all_sm)}")
print(f"  .ssc files                 : {len(all_ssc)}")

no_audio     = []
usable_pairs = []
pack_stats   = defaultdict(lambda: {'songs': 0, 'sm': 0, 'ssc': 0, 'no_audio': 0,
                                     'difficulties': defaultdict(int)})

for song_dir in song_dirs:
    pack = song_dir.parent.name
    sm_files    = list(song_dir.glob('*.sm'))
    ssc_files   = list(song_dir.glob('*.ssc'))
    audio_files = [f for f in song_dir.iterdir() if f.suffix.lower() in audio_exts]
    chart_files = sm_files if sm_files else ssc_files
    fmt = 'sm' if sm_files else 'ssc'

    pack_stats[pack]['songs'] += 1
    pack_stats[pack][fmt] += 1

    if not audio_files or not chart_files:
        pack_stats[pack]['no_audio'] += 1
        no_audio.append(song_dir)
        continue

    chart_path = str(chart_files[0])
    usable_pairs.append((str(audio_files[0]), chart_path, fmt, pack))
    for diff in quick_difficulties(chart_path):
        pack_stats[pack]['difficulties'][diff] += 1

print(f"\nSongs with audio             : {len(usable_pairs)}")
print(f"Songs missing audio (skipped): {len(no_audio)}")
if no_audio:
    for d in no_audio[:5]:
        print(f"  - {d.relative_to(root)}")
    if len(no_audio) > 5:
        print(f"  ... and {len(no_audio)-5} more")

diff_order = ['beginner', 'easy', 'medium', 'hard', 'challenge']
print(f"\n{'='*60}")
print(f"BREAKDOWN BY PACK")
print(f"{'='*60}")
for pack, stats in sorted(pack_stats.items()):
    fmt_str = []
    if stats['sm']:  fmt_str.append(f"{stats['sm']} .sm")
    if stats['ssc']: fmt_str.append(f"{stats['ssc']} .ssc")
    no_aud = f"  ⚠ {stats['no_audio']} missing audio" if stats['no_audio'] else ""
    print(f"\n  [{pack}]  {stats['songs']} songs ({', '.join(fmt_str)}){no_aud}")
    diff_counts = [f"{d[:3].upper()}:{stats['difficulties'].get(d,0)}" for d in diff_order]
    print(f"    difficulties: {' | '.join(diff_counts)}")

print(f"\n{'='*60}")
print(f"OVERALL DIFFICULTY BREAKDOWN (usable songs only)")
print(f"{'='*60}")
total_diffs = defaultdict(int)
for stats in pack_stats.values():
    for d, n in stats['difficulties'].items():
        total_diffs[d] += n
for d in diff_order:
    print(f"  {d.capitalize():<12}: {total_diffs[d]} charts")

print(f"\nTotal usable song-audio pairs: {len(usable_pairs)}")
print(f"  .sm  : {sum(1 for _,_,f,_ in usable_pairs if f=='sm')}")
print(f"  .ssc : {sum(1 for _,_,f,_ in usable_pairs if f=='ssc')}")

print(f"\n{'='*60}")
print(f"SANITY CHECK (first usable song)")
print(f"{'='*60}")
if usable_pairs:
    from utils.data_utils import parse_chart_file, build_sample
    audio_path, chart_path, fmt, pack = usable_pairs[0]
    print(f"Song  : {Path(chart_path).parent.name}  [{fmt.upper()}]")
    print(f"Pack  : {pack}")
    sm_data = parse_chart_file(chart_path)
    print(f"Title : {sm_data['title']}")
    print(f"Charts: {[(c['difficulty'], c['meter']) for c in sm_data['charts'] if 'single' in c['chart_type'].lower()]}")
    sample = build_sample(audio_path, chart_path)
    if sample:
        print(f"X shape      : {sample['X'].shape}  (timesteps, context_window, mel_bins)")
        print(f"y shape      : {sample['y'].shape}  (timesteps, 4 arrows)")
        print(f"Step density : {(sample['y'].sum(-1) > 0).mean():.1%} of timesteps have a step")
    else:
        print("build_sample returned None — no dance-single chart found")
else:
    print("No usable pairs found — check your data folder structure")

In [ ]:
# ── 4b. Build cache ───────────────────────────────────────────────────────
# Builds base_train.pkl and base_val.pkl before training starts.
# Safe to re-run — skips if cache already exists.
# Run this after cell 4 (verify data).

import os, sys, pickle
from pathlib import Path
sys.path.insert(0, '/content/ddc-chart-gen')

DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
os.makedirs(CACHE_DIR, exist_ok=True)

base_train_cache = f'{CACHE_DIR}/base_train.pkl'
base_val_cache   = f'{CACHE_DIR}/base_val.pkl'

# ── Check if cache already exists ────────────────────────────
if os.path.exists(base_train_cache) and os.path.exists(base_val_cache):
    print("Cache already exists — skipping build.")
    train_gb = os.path.getsize(base_train_cache) / 1e9
    val_gb   = os.path.getsize(base_val_cache)   / 1e9
    with open(base_train_cache, 'rb') as f:
        n = len(pickle.load(f))
    print(f"  base_train.pkl : {train_gb:.2f} GB")
    print(f"  base_val.pkl   : {val_gb:.2f} GB")
    print(f"  {n} song-difficulty samples cached")
else:
    from dataset import _process_song
    root = Path(DATA_ROOT)
    audio_exts = {'.ogg', '.mp3', '.wav'}

    # Find all usable song pairs
    all_sm  = list(root.rglob('*.sm'))
    all_ssc = list(root.rglob('*.ssc'))
    song_dirs = sorted({f.parent for f in all_sm + all_ssc})

    pairs = []
    for song_dir in song_dirs:
        sm_files    = list(song_dir.glob('*.sm'))
        ssc_files   = list(song_dir.glob('*.ssc'))
        audio_files = [f for f in song_dir.iterdir() if f.suffix.lower() in audio_exts]
        chart_files = sm_files if sm_files else ssc_files
        if chart_files and audio_files:
            pairs.append((str(audio_files[0]), str(chart_files[0])))

    print(f"Building cache for {len(pairs)} songs...")
    print("This takes ~10-20 min on first run. Progress updates every 10 songs.")

    # Process with imap_unordered so one bad song doesn't kill everything.
    # Use 2 workers — Drive I/O is the bottleneck, not CPU.
    from multiprocessing import Pool
    samples = []
    failed  = 0
    N_WORKERS = 2

    with Pool(N_WORKERS) as pool:
        for i, result in enumerate(pool.imap_unordered(_process_song, pairs)):
            samples.extend(result)
            if not result:
                failed += 1
            if (i + 1) % 10 == 0 or (i + 1) == len(pairs):
                print(f"  [{i+1}/{len(pairs)}] {len(samples)} samples built"
                      + (f"  ({failed} skipped)" if failed else ""))

    print(f"\nDone. {len(samples)} song-difficulty samples total.")
    if failed:
        print(f"  {failed} songs skipped (corrupt audio/chart or no dance-single chart)")

    # Save as both train and val base caches — same data, split happens at load time
    print("Saving to Drive...")
    with open(base_train_cache, 'wb') as f:
        pickle.dump(samples, f)
    with open(base_val_cache, 'wb') as f:
        pickle.dump(samples, f)
    print(f"Saved: {os.path.getsize(base_train_cache)/1e9:.2f} GB per file")
    print("Ready to train.")

In [ ]:
# ── 5. Train ─────────────────────────────────────────────────────────────
import os
DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
CACHE_DIR = '/content/drive/MyDrive/142B Group/ddc_project/data/cache'
CKPT_DIR  = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['DATA_ROOT'] = DATA_ROOT
os.environ['CACHE_DIR'] = CACHE_DIR
os.environ['CKPT_DIR']  = CKPT_DIR

%cd /content/ddc-chart-gen
!python train.py \
    --data_root      "$DATA_ROOT" \
    --cache_dir      "$CACHE_DIR" \
    --checkpoint_dir "$CKPT_DIR" \
    --num_workers 0

In [ ]:
# ── 5b. Resume after disconnect ──────────────────────────────────────────
import torch
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
print(f"Last saved: stage {ckpt['stage']}, epoch {ckpt['epoch']}, val F1 {ckpt['val_f1']:.4f}")
print(f"Rerun cell 5 — curriculum_start is read from config.py (currently set to {ckpt['stage']})")

In [ ]:
# ── 6. Plot loss curves ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import json
import matplotlib.pyplot as plt

with open('logs/train_history.json') as f:
    history = json.load(f)

train_h = history['train']
val_h   = history['val']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
stage_colors = {0:'#3498db', 1:'#2ecc71', 2:'#f39c12', 3:'#e74c3c', 4:'#9b59b6'}
stage_names  = {0:'Beginner', 1:'Easy', 2:'Medium', 3:'Hard', 4:'Challenge'}

for metric, ax, title in [
    ('loss',       axes[0], 'Total Loss'),
    ('step_loss',  axes[1], 'Step Placement Loss'),
    ('f1',         axes[2], 'Step F1 Score'),
]:
    for stage in range(5):
        t_vals = [x[metric] for x in train_h if x['stage'] == stage]
        v_vals = [x[metric] for x in val_h   if x['stage'] == stage]
        if not t_vals:
            continue
        color = stage_colors[stage]
        ax.plot(t_vals, color=color, alpha=0.9, label=f'{stage_names[stage]} train')
        ax.plot(v_vals, color=color, alpha=0.5, linestyle='--', label=f'{stage_names[stage]} val')
    ax.set_title(title)
    ax.set_xlabel('Epoch within stage')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Training curves by curriculum stage', y=1.02)
plt.tight_layout()
plt.savefig('logs/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/training_curves.png /content/drive/MyDrive/ddc_project/logs/
print('Saved to Drive.')

In [ ]:
# ── 7. Generate a chart ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
import os
from pathlib import Path

# Option A: pick a song from the dataset (uncomment)
# DATA_ROOT = '/content/drive/MyDrive/142B Group/ddc_project/data/raw'
# all_charts = list(Path(DATA_ROOT).rglob('*.sm')) + list(Path(DATA_ROOT).rglob('*.ssc'))
# for chart in all_charts:
#     audio = [f for f in chart.parent.iterdir()
#              if f.suffix.lower() in {'.ogg', '.mp3', '.wav'}]
#     if audio:
#         test_audio = str(audio[0])
#         print(f'Using: {chart.parent.name} — {audio[0].name}')
#         break

# Option B: upload any song from your computer (mp3, ogg, wav)
from google.colab import files as colab_files
uploaded = colab_files.upload()
test_audio = list(uploaded.keys())[0]
print(f'Using: {test_audio}')

# ── Set BPM manually — check the song's .sm/.ssc file or use a BPM finder ──
SONG_BPM = 123.0   # <-- edit this

CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'
os.environ['TEST_AUDIO'] = test_audio
os.environ['CKPT_DIR']   = CKPT_DIR
os.environ['SONG_BPM']   = str(SONG_BPM)

# Adjust --difficulty (0-4) and --threshold (0.1-0.9, lower = more arrows)
!python generate.py \
    --audio      "$TEST_AUDIO" \
    --checkpoint "$CKPT_DIR/best_model.pt" \
    --bpm        "$SONG_BPM" \
    --difficulty 2 \
    --threshold  0.5 \
    --output     output_chart

!cp -r output_chart '/content/drive/MyDrive/142B Group/ddc_project/'
print('Done! Run cell 9 to download visualizer.html')

In [ ]:
# ── 8. Threshold sweep ───────────────────────────────────────────────────
# Shows how the step threshold knob controls chart density.
# Lower threshold = more arrows = harder feel.
# Higher threshold = fewer arrows = easier feel.

%cd /content/ddc-chart-gen
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/ddc-chart-gen')

from models.model import DDRTransformer, generate_chart
from generate import audio_to_model_input

CKPT_DIR = '/content/drive/MyDrive/ddc_project/checkpoints'
ckpt = torch.load(f'{CKPT_DIR}/best_model.pt', map_location='cpu')
model_args = ckpt.get('args', {})
model = DDRTransformer(
    d_model=model_args.get('d_model', 256),
    nhead=model_args.get('nhead', 8),
    num_encoder_layers=model_args.get('n_layers', 4),
    dim_feedforward=model_args.get('d_ff', 1024),
    dropout=0.0,
)
model.load_state_dict(ckpt['model_state'])
X = audio_to_model_input(test_audio)

thresholds = np.linspace(0.1, 0.9, 17)
densities  = []
for th in thresholds:
    mask, _ = generate_chart(model, X, difficulty=2, step_threshold=float(th))
    densities.append(mask.mean())

plt.figure(figsize=(7, 4))
plt.plot(thresholds, densities, 'o-', color='#e74c3c')
plt.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='default threshold')
plt.xlabel('Step threshold')
plt.ylabel('Fraction of timesteps with a step')
plt.title('Difficulty knob: threshold vs. chart density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('logs/threshold_sweep.png', dpi=150)
plt.show()
!mkdir -p /content/drive/MyDrive/ddc_project/logs
!cp logs/threshold_sweep.png /content/drive/MyDrive/ddc_project/logs/

In [ ]:
# ── 9. Download results ──────────────────────────────────────────────────
%cd /content/ddc-chart-gen
CKPT_DIR = '/content/drive/MyDrive/142B Group/ddc_project/checkpoints'

from google.colab import files
files.download('output_chart/visualizer.html')
files.download('output_chart/chart.sm')
files.download(f'{CKPT_DIR}/best_model.pt')
files.download('logs/training_curves.png')
files.download('logs/threshold_sweep.png')